In [1]:
import cv2
import json
import os
import pandas as pd
from cv_bridge import CvBridge
from bagpy import bagreader
import ast
import numpy as np

In [2]:
# Specify the path to your ROS bag file
bag_path = 'bag.bag'

# Create an instance of bagreader
bag = bagreader(bag_path)

# Print the columns of the topic table to understand its structure
print("Columns in topic_table:", bag.topic_table.columns)

# Assuming the correct column for topics is named 'Topics'
topics = bag.topic_table['Topics']  # Adjust 'Topics' if it's named differently

# Now, iterate over each topic and retrieve messages
for topic in topics:
    messages = bag.message_by_topic(topic)
    print(f"Messages from '{topic}' : {messages}")

[INFO]  Successfully created the data folder bag.
Columns in topic_table: Index(['Topics', 'Types', 'Message Count', 'Frequency'], dtype='object')
Messages from '/camera/aligned_depth_to_color/camera_info' : bag/camera-aligned_depth_to_color-camera_info.csv
Messages from '/camera/aligned_depth_to_color/image_raw' : bag/camera-aligned_depth_to_color-image_raw.csv
Messages from '/camera/camera_pose' : bag/camera-camera_pose.csv
Messages from '/camera/color/image_rect_color' : bag/camera-color-image_rect_color.csv
Messages from '/camera/joint_states' : bag/camera-joint_states.csv


In [4]:
# Initialize CvBridge
bridge = CvBridge()

# Retrieve messages from the specified topic
message = bag.message_by_topic('/camera/color/image_rect_color')
data = pd.read_csv(message)

# Convert the string representation of bytes directly into a byte array
image_bytes = ast.literal_eval(data.iloc[0]['data'])

# Reshape the byte array to an image array using the dimensions and step provided in the CSV
height, width, step = data.iloc[0]['height'], data.iloc[0]['width'], data.iloc[0]['step']
image_array = np.frombuffer(image_bytes, dtype=np.uint8).reshape((height, width, 3))

# Display the image using OpenCV
cv2.imwrite('camera_extraced_data/Extracted_Image.jpg', image_array)

True

In [11]:
import cv2
import numpy as np
import os
import pandas as pd
import ast
from cv_bridge import CvBridge

# Initialize CvBridge
bridge = CvBridge()

# Retrieve messages from the specified topic (assuming bag object and message retrieval method)
message = bag.message_by_topic('/camera/aligned_depth_to_color/image_raw')

# Load data from the message into a Pandas DataFrame
data = pd.read_csv(message)

# Convert the string representation of bytes into a byte array
image_bytes = ast.literal_eval(data.iloc[0]['data'])

# Retrieve the height, width, and step from the CSV
height, width, step = int(data.iloc[0]['height']), int(data.iloc[0]['width']), int(data.iloc[0]['step'])

# If it's a depth image, it's likely single-channel (grayscale), reshape accordingly
# Assuming the depth image is 16-bit (change to np.uint8 if 8-bit)
image_array = np.frombuffer(image_bytes, dtype=np.uint8).reshape((height, width))

# Optionally normalize the depth values for display (for visualization)
# This step scales the depth values to the range of 0-255 for display as an 8-bit image
image_normalized = cv2.normalize(image_array, None, 0, 255, cv2.NORM_MINMAX)
image_normalized = np.uint8(image_normalized)

# Ensure the directory exists
output_dir = 'camera_extracted_data/'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Save the image as a JPG using OpenCV
image_path = os.path.join(output_dir, 'Extracted_Depth_Image.jpg')
success = cv2.imwrite(image_path, image_normalized)

# Check if the image was saved successfully
if success:
    print(f"Image saved successfully at {image_path}")
else:
    print("Failed to save the image.")

# Save the raw NumPy array (depth values) as a CSV
csv_path = os.path.join(output_dir, 'Extracted_Depth_Image_Array.csv')
np.savetxt(csv_path, image_array, delimiter=",", fmt='%d')

print(f"NumPy array saved successfully as CSV at {csv_path}")



Image saved successfully at camera_extracted_data/Extracted_Depth_Image.jpg
NumPy array saved successfully as CSV at camera_extracted_data/Extracted_Depth_Image_Array.csv


In [5]:
def create_transformation_matrices(df):

    transformation_matrices = []

    row = df.iloc[0]
        
    # Extract quaternion and convert to rotation matrix
    quaternion = [row['orientation.x'], row['orientation.y'], row['orientation.z'], row['orientation.w']]
    
    x, y, z, w = quaternion
    
    rotation_matrix = np.array([
            [1 - 2*y**2 - 2*z**2, 2*x*y - 2*z*w,       2*x*z + 2*y*w],
            [2*x*y + 2*z*w,       1 - 2*x**2 - 2*z**2, 2*y*z - 2*x*w],
            [2*x*z - 2*y*w,       2*y*z + 2*x*w,       1 - 2*x**2 - 2*y**2]
        ])

    # Create the transformation matrix
    transform = np.zeros((4, 4))
    transform[:3, :3] = rotation_matrix
    transform[:3, 3] = row[['position.x', 'position.y', 'position.z']]
    transform[3, 3] = 1

    # Append the matrix to the list
    transformation_matrices.append(transform)
            
    new_df = pd.DataFrame(transformation_matrices[0])
    return new_df

In [6]:

pose=bag.message_by_topic('/camera/camera_pose')
camera_camera_pose_df = pd.read_csv(pose)


new_df = create_transformation_matrices(camera_camera_pose_df)



new_df.to_csv("camera_extraced_data/transformed_camera_pose.csv", index=False, header=False)


In [22]:
import cv2
import os
import pandas as pd
from cv_bridge import CvBridge
from bagpy import bagreader
import ast
import numpy as np

# Specify the path to your ROS bag folder
bag_folder_path = 'fall_24_data'  # Change this to your folder containing .bag files

# Ensure the output directory exists
if not os.path.exists(bag_folder_path):
    os.makedirs(bag_folder_path)

# List all .bag files in the specified folder
bag_files = [f for f in os.listdir(bag_folder_path) if f.endswith('.bag')]

# Initialize a counter for naming the output folders
counter = 1

# Loop through each bag file
for bag_file in bag_files:
    bag_path = os.path.join(bag_folder_path, bag_file)
    
    # Create an instance of bagreader
    bag = bagreader(bag_path)
    
    # Print the columns of the topic table to understand its structure
    print(f"Processing {bag_file}...")
    print("Columns in topic_table:", bag.topic_table.columns)

    # Assuming the correct column for topics is named 'Topics'
    topics = bag.topic_table['Topics']  # Adjust 'Topics' if it's named differently

    # Create a folder for the current pose
    pose_folder = os.path.join(bag_folder_path, f'pose_{counter}')
    os.makedirs(pose_folder, exist_ok=True)

    # Iterate over each topic and retrieve messages
    for topic in topics:
        messages = bag.message_by_topic(topic)
        print(f"Messages from '{topic}' : {messages}")

    # Initialize CvBridge
    bridge = CvBridge()

    # Retrieve messages from the specified topic for color images
    message = bag.message_by_topic('/camera/color/image_rect_color')
    data = pd.read_csv(message)

    # Convert the string representation of bytes directly into a byte array
    image_bytes = ast.literal_eval(data.iloc[0]['data'])

    # Reshape the byte array to an image array using the dimensions and step provided in the CSV
    height, width, step = data.iloc[0]['height'], data.iloc[0]['width'], data.iloc[0]['step']
    image_array = np.frombuffer(image_bytes, dtype=np.uint8).reshape((height, width, 3))

    # Convert the image from BGR to RGB
    image_array_rgb = cv2.cvtColor(image_array, cv2.COLOR_BGR2RGB)

    # Save the RGB image using OpenCV
    item_image_path = os.path.join(pose_folder, 'item_image.png')
    cv2.imwrite(item_image_path, image_array_rgb)

    # Load depth image data
    message = bag.message_by_topic('/camera/aligned_depth_to_color/image_raw')
    data = pd.read_csv(message)

    # Convert the string representation of bytes into a byte array
    image_bytes = ast.literal_eval(data.iloc[0]['data'])

    # Retrieve the height, width, and step from the CSV
    height, width, step = int(data.iloc[0]['height']), int(data.iloc[0]['width']), int(data.iloc[0]['step'])

    # Assuming the depth image is 16-bit (change to np.uint16 if 8-bit)
    image_array = np.frombuffer(image_bytes, dtype=np.uint16).reshape((height, width))

    # Normalize the depth values for display
    image_normalized = cv2.normalize(image_array, None, 0, 255, cv2.NORM_MINMAX)
    image_normalized = np.uint8(image_normalized)

    # Save the normalized depth image
    depth_image_path = os.path.join(pose_folder, 'depth_image_pixel_transform.png')
    success = cv2.imwrite(depth_image_path, image_normalized)

    # Check if the image was saved successfully
    if success:
        print(f"Depth image saved successfully at {depth_image_path}")
    else:
        print("Failed to save the depth image.")

    # Save the raw NumPy array (depth values) as a CSV
    depth_array_path = os.path.join(pose_folder, 'depth_array.csv')
    np.savetxt(depth_array_path, image_array, delimiter=",", fmt='%d')

    print(f"NumPy array saved successfully as CSV at {depth_array_path}")

    # Define the function to create transformation matrices
    def create_transformation_matrices(df):
        transformation_matrices = []
        row = df.iloc[0]
        
        # Extract quaternion and convert to rotation matrix
        quaternion = [row['orientation.x'], row['orientation.y'], row['orientation.z'], row['orientation.w']]
        
        x, y, z, w = quaternion
        
        rotation_matrix = np.array([
                [1 - 2*y**2 - 2*z**2, 2*x*y - 2*z*w,       2*x*z + 2*y*w],
                [2*x*y + 2*z*w,       1 - 2*x**2 - 2*z**2, 2*y*z - 2*x*w],
                [2*x*z - 2*y*w,       2*y*z + 2*x*w,       1 - 2*x**2 - 2*y**2]
            ])

        # Create the transformation matrix
        transform = np.zeros((4, 4))
        transform[:3, :3] = rotation_matrix
        transform[:3, 3] = row[['position.x', 'position.y', 'position.z']]
        transform[3, 3] = 1

        # Append the matrix to the list
        transformation_matrices.append(transform)
                
        new_df = pd.DataFrame(transformation_matrices[0])
        return new_df

    # Retrieve camera pose messages
    pose = bag.message_by_topic('/camera/camera_pose')
    camera_camera_pose_df = pd.read_csv(pose)

    # Create transformation matrices
    new_df = create_transformation_matrices(camera_camera_pose_df)

    # Save the transformation matrix as a CSV
    transformation_csv_path = os.path.join(pose_folder, 'camera_pose.csv')
    new_df.to_csv(transformation_csv_path, index=False, header=False)

    print(f"Transformation matrix saved successfully as CSV at {transformation_csv_path}")

    # Increment the counter for the next bag file
    counter += 1

print("Processing complete for all bag files.")


[INFO]  Successfully created the data folder fall_24_data\bag1.
Processing bag1.bag...
Columns in topic_table: Index(['Topics', 'Types', 'Message Count', 'Frequency'], dtype='object')
Messages from '/camera/aligned_depth_to_color/camera_info' : fall_24_data\bag1/camera-aligned_depth_to_color-camera_info.csv
Messages from '/camera/aligned_depth_to_color/image_raw' : fall_24_data\bag1/camera-aligned_depth_to_color-image_raw.csv
Messages from '/camera/camera_pose' : fall_24_data\bag1/camera-camera_pose.csv
Messages from '/camera/color/image_rect_color' : fall_24_data\bag1/camera-color-image_rect_color.csv
Messages from '/camera/joint_states' : fall_24_data\bag1/camera-joint_states.csv
Depth image saved successfully at fall_24_data\pose_1\depth_image_pixel_transform.png
NumPy array saved successfully as CSV at fall_24_data\pose_1\depth_array.csv
Transformation matrix saved successfully as CSV at fall_24_data\pose_1\camera_pose.csv
[INFO]  Successfully created the data folder fall_24_data\b